In [ ]:
!pip install -q segmentation_models_pytorch

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q3-stage3-2026")

print("Path to dataset files:", path)

In [ ]:
from torch.utils.data import DataLoader,Dataset, random_split

In [ ]:
def remap_mask(mask):
    # Remaps a mask's pixel values to a consecutive range starting at 0
    mask = mask.long()
    unique_values = torch.unique(mask)
    remapped_mask = torch.zeros_like(mask)

    for new_val, old_val in enumerate(sorted(unique_values.tolist())):
        remapped_mask[mask == old_val] = new_val

    return remapped_mask

In [ ]:
# TO DO
import os
import pandas as pd
from PIL import Image
import torch
import torchvision.transforms as transforms
from torch.utils.data import Dataset
import numpy as np
# Custom Dataset Class
import os
import glob
from torch.utils.data import Dataset
from PIL import Image
import torchvision.transforms as transforms

class SegmentDataset(Dataset):
    def __init__(self, image_dir, mask_dir, transform=None, target_transform=None):
        self.image_paths = glob.glob(os.path.join(image_dir, "*.jpg"))  # Get all image paths
        self.mask_paths = glob.glob(os.path.join(mask_dir, "*.png"))  # Get all mask paths

        self.image_paths.sort()
        self.mask_paths.sort()

        self.transform = transform
        self.target_transform = target_transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        # 🔹 Load the image and mask
        image = Image.open(self.image_paths[idx]).convert("RGB")
        mask = Image.open(self.mask_paths[idx]).convert("L")

        # 🔹 Apply transformations for image
        if self.transform:
            image = self.transform(image)

        # 🔹 Apply transformations for mask
        if self.target_transform:
            mask = self.target_transform(mask)

        mask = remap_mask(mask)

        return image, mask  # Return image-mask pair

In [ ]:
from torch.utils.data import DataLoader
from torch import nn

# Define transforms for images and masks
image_transforms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((256, 256)),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # Standard ImageNet normalization
])

mask_transforms = transforms.Compose([
    transforms.Resize((256, 256), interpolation=transforms.InterpolationMode.NEAREST),  # Keep segmentation masks intact
    transforms.PILToTensor(),
])

In [ ]:
image_dir = os.path.join(path, "dataset", "images")
mask_dir = os.path.join(path, "dataset", "masks")

dataset = SegmentDataset(image_dir,mask_dir, image_transforms, mask_transforms)

train_len = int(len(dataset) * 0.8)
val_len = len(dataset) - train_len

train_dataset, val_dataset = random_split(dataset, [train_len, val_len])


train_loader = DataLoader(train_dataset, 32, True)
test_loader = DataLoader(val_dataset, 32, False)

print(len(dataset))
print(len(train_dataset))

In [ ]:
import matplotlib.pyplot as plt

img, mask = next(iter(test_loader))

fig, ax = plt.subplots(5,2, figsize=(10,15))

for i in range(5):
    ax[i,0].imshow(img[i].permute(1,2,0))
    ax[i,1].imshow(mask[i].squeeze(0), cmap='gray')

    print(torch.unique(mask[1].squeeze(0)))

In [ ]:
# TO DO
import segmentation_models_pytorch as smp

# Define U-Net Model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = smp.Unet(
    encoder_name="efficientnet-b1",  # Pretrained encoder (backbone)
    encoder_weights="imagenet",  # Use ImageNet weights
    in_channels=3,  # RGB images
    classes=8,
).to(device)

In [ ]:
from tqdm import tqdm

In [ ]:
# TO DO
def train(model, optimizer, criterion, train_loader, device):
    model.train()
    total_loss = 0.0

    for img, mask in tqdm(train_loader):
        img = img.to(device)
        mask = mask.squeeze(dim=1).to(device)

        outputs = model(img)

        loss = criterion(outputs, mask)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(train_loader)

def validate(model, criterion, test_loader, device):
    model.eval()
    total_loss = 0.0

    with torch.no_grad():
        for img, mask in tqdm(test_loader):
            img = img.to(device)
            mask = mask.squeeze(dim=1).to(device)

            outputs = model(img)
            loss = criterion(outputs, mask)
            total_loss += loss.item()

    return total_loss / len(test_loader)

In [ ]:
# TO DO
loss = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001)

train_losses = []
val_losses = []

e = 20

# Training Loop
for epoch in range(e):
    train_loss = train(model, optimizer, loss, train_loader, device)
    val_loss = validate(model, loss, test_loader, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f"Epoch {epoch+1}/{e}: Train Loss = {train_loss:.4f}, Val Loss = {val_loss:.4f}")

In [ ]:
plt.plot(range(1, e+1), train_losses, label="Train Loss", marker='o')
plt.plot(range(1, e+1), val_losses, label="Validation Loss", marker='o')
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.show()

In [ ]:
# TO DO
import random
model.eval()

test_samples = random.sample(range(len(val_dataset)), 5)

for i in test_samples:
  img, mask = val_dataset[i]

  with torch.no_grad():
    pred_mask = model(img.unsqueeze(0).to(device))
    pred_mask = torch.argmax(pred_mask, dim=1).squeeze().cpu()

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))


    axes[0].imshow(img.permute(1,2,0))
    axes[0].set_title("Original Image")
    axes[0].axis("off")

    axes[1].imshow(mask.squeeze())
    axes[1].set_title("Ground Truth Mask")
    axes[1].axis("off")

    axes[2].imshow(pred_mask)
    axes[2].set_title("Predicted Mask")
    axes[2].axis("off")

    plt.show()